In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Load data

In [4]:
df = pd.read_csv(r"data\WA_Fn-UseC_-Telco-Customer-Churn.csv")

## Prepare Data

##### Column Name Standardization

In [5]:
df.columns = df.columns.str.lower()
columns = list(df.columns)

##### Column Type Standardization

In [6]:
df["seniorcitizen"] = df["seniorcitizen"].astype(str).str.replace('0', 'No').str.replace('1', 'Yes')

df["totalcharges"] = pd.to_numeric(df["totalcharges"], errors = "coerce" )

df["churn"] = df["churn"].str.replace("No", "0").str.replace("Yes", "1").astype(int)

##### Categorical & Numerical Columns

In [7]:
categorical_columns = list(df.dtypes[df.dtypes == 'str'].index)
categorical_columns.remove("customerid")

numerical_columns = list(df.dtypes[(df.dtypes == 'int64') | (df.dtypes == 'float64')].index)
numerical_columns.remove("churn")

##### Standardizing Categorial Rows

In [8]:
for col in categorical_columns:
    df[col] = df[col].str.lower().str.replace(' ', '_').str.replace('-', '_')

##### Addressing Null Values

In [9]:
df["totalcharges"] = df["totalcharges"].fillna(df["totalcharges"].mean())
print(df.isna().sum())

customerid          0
gender              0
seniorcitizen       0
partner             0
dependents          0
tenure              0
phoneservice        0
multiplelines       0
internetservice     0
onlinesecurity      0
onlinebackup        0
deviceprotection    0
techsupport         0
streamingtv         0
streamingmovies     0
contract            0
paperlessbilling    0
paymentmethod       0
monthlycharges      0
totalcharges        0
churn               0
dtype: int64


## Validation Framwork

In [10]:
from sklearn.model_selection import train_test_split

In [11]:
df_full_train, df_test = train_test_split(df, test_size = 0.2, random_state=1)

df_train, df_val = train_test_split(df_full_train, test_size = 0.25)

df_test.reset_index(drop = True, inplace = True)
df_train.reset_index(drop = True, inplace = True)
df_val.reset_index(drop = True, inplace = True)

y_test = df_test['churn'].values 
y_train = df_train['churn'].values 
y_val = df_val['churn'].values 

del df_test['churn']
del df_train['churn']
del df_val['churn']


## EDA

In [12]:
df_full_train = df_full_train.reset_index(drop = True)
df_full_train.head()

,customerid,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,...,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn
0,5442-PPTJY,male,no,yes,yes,12,yes,no,no,no_internet_service,...,no_internet_service,no_internet_service,no_internet_service,no_internet_service,two_year,no,mailed_check,19.70,258.35,0
1,6261-RCVNS,female,no,no,no,42,yes,no,dsl,yes,...,yes,yes,no,yes,one_year,no,credit_card_(automatic),73.90,3160.55,1
2,2176-OSJUV,male,no,yes,no,71,yes,yes,dsl,yes,...,no,yes,no,no,two_year,no,bank_transfer_(automatic),65.15,4681.75,0
3,6161-ERDGD,male,no,yes,yes,71,yes,yes,dsl,yes,...,yes,yes,yes,yes,one_year,no,electronic_check,85.45,6300.85,0
4,2364-UFROM,male,no,no,no,30,yes,no,dsl,yes,...,no,yes,yes,no,one_year,no,electronic_check,70.40,2044.75,0


In [13]:
df_full_train['churn'].value_counts(normalize = True)
churn_rate = df_full_train["churn"].mean()

We can conclude that the global chrun rate sits around 27% of users. Let's look at the churn rate by groups. 

In [14]:
churn_females = df[df["gender"] == "female"]["churn"].mean()
churn_male = df[df["gender"] == "male"]["churn"].mean()

print(churn_females, churn_male, churn_rate)

0.26920871559633025 0.2616033755274262 0.26996805111821087


##### Churn Rate, Absolute Risk, and Relative Risk

In [15]:
from IPython.display import display


for c in categorical_columns:
    df_group = df.groupby(c)["churn"].agg(["mean", "count"])
    df_group["difference"] = df_group["mean"] - churn_rate
    df_group["risk"] = df_group["mean"] / churn_rate
    df_group["mean"] = df_group["mean"] * 100


    display(df_group)

,mean,count,difference,risk
gender,,,,
female,26.920872,3488,-0.000759,0.997187
male,26.160338,3555,-0.008365,0.969016


,mean,count,difference,risk
seniorcitizen,,,,
no,23.606168,5901,-0.033906,0.874406
yes,41.681261,1142,0.146845,1.543933


,mean,count,difference,risk
partner,,,,
no,32.957979,3641,0.059612,1.220810
yes,19.664903,3402,-0.073319,0.728416


,mean,count,difference,risk
dependents,,,,
no,31.279140,4933,0.042823,1.158624
yes,15.450237,2110,-0.115466,0.572299


,mean,count,difference,risk
phoneservice,,,,
no,24.926686,682,-0.020701,0.923320
yes,26.709637,6361,-0.002872,0.989363


...

,mean,count,difference,risk
streamingtv,,,,
no,33.523132,2810,0.065263,1.241744
no_internet_service,7.404980,1526,-0.195918,0.274291
yes,30.070188,2707,0.030734,1.113842


,mean,count,difference,risk
streamingmovies,,,,
no,33.680431,2785,0.066836,1.247571
no_internet_service,7.404980,1526,-0.195918,0.274291
yes,29.941435,2732,0.029446,1.109073


,mean,count,difference,risk
contract,,,,
month_to_month,42.709677,3875,0.157129,1.582027
one_year,11.269518,1473,-0.157273,0.417439
two_year,2.831858,1695,-0.241649,0.104896


,mean,count,difference,risk
paperlessbilling,,,,
no,16.330084,2872,-0.106667,0.604889
yes,33.565092,4171,0.065683,1.243299


,mean,count,difference,risk
paymentmethod,,,,
bank_transfer_(automatic),16.709845,1544,-0.102870,0.618956
credit_card_(automatic),15.243101,1522,-0.117537,0.564626
electronic_check,45.285412,2365,0.182886,1.677436
mailed_check,19.106700,1612,-0.078901,0.707739


##### Mutual Information

In [ ]:
from sklearn.metrics import mutual_info_score

def churn_mutual_info_score(col):
    return mutual_info_score(col, df_full_train["churn"])

mi = df_full_train[categorical_columns].apply(churn_mutual_info_score)

mi = mi.sort_values(ascending=False).to_frame(name = "mi")


,mi
contract,0.098320
onlinesecurity,0.063085
techsupport,0.061032
internetservice,0.055868
onlinebackup,0.046923
deviceprotection,0.043453
paymentmethod,0.043210
streamingtv,0.031853
streamingmovies,0.031581
paperlessbilling,0.017589


Based on mutual information between Churn and categorical variables, we can conclude that the most important categorical variable is contract. Following contract is: Online Security, and Tech Support.

##### Correlation

In [46]:
df_full_train[numerical_columns].corrwith(df_full_train["churn"]).to_frame(name = 'r')

,r
tenure,-0.351885
monthlycharges,0.196805
totalcharges,-0.197365


As churn rate increases we can observe that tenure decreases. So longer the tenure is related to lower the churn rate.

In [55]:
print(f"Tenure [0,24]: {df_full_train[(df_full_train["tenure"] <= 24)].churn.mean()}")
print(f"Tenure (24,48]: {df_full_train[(df_full_train["tenure"] > 24) & (df_full_train["tenure"] <= 48)].churn.mean()}")
print(f"Tenure (48,72]: {df_full_train[(df_full_train["tenure"] > 48) & (df_full_train["tenure"] <= 72)].churn.mean()}")

Tenure [0,24]: 0.420909444228527
Tenure (24,48]: 0.20465116279069767
Tenure (48,72]: 0.09824957651044608


One Hot Encoding

In [61]:
df_full_train[categorical_columns].head(10)

,gender,seniorcitizen,partner,dependents,phoneservice,multiplelines,internetservice,onlinesecurity,onlinebackup,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod
0,male,no,yes,yes,yes,no,no,no_internet_service,no_internet_service,no_internet_service,no_internet_service,no_internet_service,no_internet_service,two_year,no,mailed_check
1,female,no,no,no,yes,no,dsl,yes,yes,yes,yes,no,yes,one_year,no,credit_card_(automatic)
2,male,no,yes,no,yes,yes,dsl,yes,yes,no,yes,no,no,two_year,no,bank_transfer_(automatic)
3,male,no,yes,yes,yes,yes,dsl,yes,no,yes,yes,yes,yes,one_year,no,electronic_check
4,male,no,no,no,yes,no,dsl,yes,yes,no,yes,yes,no,one_year,no,electronic_check
5,female,no,yes,yes,yes,no,dsl,yes,yes,yes,yes,no,no,month_to_month,no,mailed_check
6,male,no,yes,no,yes,yes,fiber_optic,yes,yes,yes,no,no,yes,two_year,yes,electronic_check
7,male,no,no,no,yes,no,fiber_optic,no,no,yes,yes,no,yes,month_to_month,no,electronic_check
8,male,yes,yes,no,yes,yes,fiber_optic,no,no,yes,no,no,no,month_to_month,yes,electronic_check
9,female,yes,yes,yes,yes,no,fiber_optic,no,no,no,no,no,no,month_to_month,no,bank_transfer_(automatic)


In [ ]:
from sklearn.feature_extraction import DictVectorizer

columns_to_be_encoded = list(df_full_train[categorical_columns].columns[(df_full_train[categorical_columns].nunique() > 2)])
dict = df_full_train[columns_to_be_encoded].to_dict(orient = "records")


#ohe = DictVectorizer()


[{'multiplelines': 'no',
  'internetservice': 'no',
  'onlinesecurity': 'no_internet_service',
  'onlinebackup': 'no_internet_service',
  'deviceprotection': 'no_internet_service',
  'techsupport': 'no_internet_service',
  'streamingtv': 'no_internet_service',
  'streamingmovies': 'no_internet_service',
  'contract': 'two_year',
  'paymentmethod': 'mailed_check'},
 {'multiplelines': 'no',
  'internetservice': 'dsl',
  'onlinesecurity': 'yes',
  'onlinebackup': 'yes',
  'deviceprotection': 'yes',
  'techsupport': 'yes',
  'streamingtv': 'no',
  'streamingmovies': 'yes',
  'contract': 'one_year',
  'paymentmethod': 'credit_card_(automatic)'},
 {'multiplelines': 'yes',
  'internetservice': 'dsl',
  'onlinesecurity': 'yes',
  'onlinebackup': 'yes',
  'deviceprotection': 'no',
  'techsupport': 'yes',
  'streamingtv': 'no',
  'streamingmovies': 'no',
  'contract': 'two_year',
  'paymentmethod': 'bank_transfer_(automatic)'},
 {'multiplelines': 'yes',
  'internetservice': 'dsl',
  'onlinesecur